In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import plotly.express as px
import random

def generate_synthetic_data(num_samples=100, size=32):
    images = []
    labels = []

    for _ in range(num_samples):
        img_v = np.zeros((size, size))
        img_v[:, ::2] = 1.0
        img_v += np.random.normal(0, 0.2, (size, size))
        images.append(img_v)
        labels.append(0)

        img_h = np.zeros((size, size))
        img_h[::2, :] = 1.0
        img_h += np.random.normal(0, 0.2, (size, size))
        images.append(img_h)
        labels.append(1)

        img_c = np.zeros((size, size))
        img_c[::2, ::2] = 1.0
        img_c[1::2, 1::2] = 1.0
        img_c += np.random.normal(0, 0.2, (size, size))
        images.append(img_c)
        labels.append(2)

    images = torch.tensor(np.array(images), dtype=torch.float32).unsqueeze(1)
    labels = torch.tensor(labels, dtype=torch.long)
    return images, labels

images, labels = generate_synthetic_data(num_samples=333)
class_names = {0: 'Lignes Verticales', 1: 'Lignes Horizontales', 2: 'Damier'}
labels_text = [class_names[l.item()] for l in labels]

print(f" Données générées : {len(images)} images de taille 8x8.")

class TinyEmbedder(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 32, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = TinyEmbedder()
optimizer = optim.Adam(model.parameters(), lr=0.01)
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

EPOCHS = 20
history_df = []

print(" Début de l'entraînement Metric Learning...")

for epoch in range(EPOCHS + 1):

    model.eval()
    with torch.no_grad():
        embeddings = model(images).numpy()

    for i in range(len(embeddings)):
        history_df.append({
            'Epoch': f"Époque {epoch}",
            'X': embeddings[i, 0],
            'Y': embeddings[i, 1],
            'Classe': labels_text[i]
        })

    if epoch == EPOCHS:
        break

    model.train()
    optimizer.zero_grad()

    anchors, positives, negatives = [], [], []
    for _ in range(150):
        anchor_idx = random.randint(0, len(images) - 1)
        anchor_label = labels[anchor_idx].item()

        pos_idx = random.choice([i for i, l in enumerate(labels) if l.item() == anchor_label])
        neg_idx = random.choice([i for i, l in enumerate(labels) if l.item() != anchor_label])

        anchors.append(images[anchor_idx])
        positives.append(images[pos_idx])
        negatives.append(images[neg_idx])

    anchors = torch.stack(anchors)
    positives = torch.stack(positives)
    negatives = torch.stack(negatives)

    emb_a = model(anchors)
    emb_p = model(positives)
    emb_n = model(negatives)

    loss = triplet_loss(emb_a, emb_p, emb_n)
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Époque {epoch}/{EPOCHS} - Triplet Loss: {loss.item():.4f}")

print(" Entraînement terminé !")

df_plot = pd.DataFrame(history_df)

x_max, x_min = df_plot['X'].max(), df_plot['X'].min()
y_max, y_min = df_plot['Y'].max(), df_plot['Y'].min()
padding = 1.0

fig = px.scatter(
    df_plot,
    x='X', y='Y',
    color='Classe',
    animation_frame='Epoch',
    title="Évolution de l'Espace Latent (Metric Learning)",
    range_x=[x_min - padding, x_max + padding],
    range_y=[y_min - padding, y_max + padding],
    opacity=0.8
)

fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500

fig.show()

In [ ]:
import matplotlib.pyplot as plt

def visualize_synthetic_data(images, labels, num_samples_per_class=4):
    print(" Affichage de quelques exemples de données synthétiques...")
    plt.figure(figsize=(10, 6))

    for class_idx in range(3):
        idx_list = [i for i, label in enumerate(labels) if label.item() == class_idx]

        selected_idx = random.sample(idx_list, num_samples_per_class)

        for i, idx in enumerate(selected_idx):
            plt.subplot(3, num_samples_per_class, class_idx * num_samples_per_class + i + 1)

            img_to_show = images[idx].squeeze().numpy()

            plt.imshow(img_to_show, cmap='gray')

            if i == 0:
                plt.ylabel(class_names[class_idx], fontsize=10, fontweight='bold')

            plt.xticks([])
            plt.yticks([])

    plt.suptitle("Aperçu des données synthétiques (8x8 pixels)", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_synthetic_data(images, labels)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import random

SEED = 123
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Exécution sur : {device}")

class SimpleClassifier3D(nn.Module):
    def __init__(self, num_classes=3, embedding_dim=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.flatten = nn.Flatten()
        self.embedding_fc = nn.Linear(8, embedding_dim)
        self.classifier_fc = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.conv(x)
        features = self.flatten(features)
        embedding = self.embedding_fc(features)
        logits = self.classifier_fc(embedding)
        return logits, embedding
    
model = SimpleClassifier3D().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 20
history_df = []
images, labels = images.to(device), labels.to(device)

print(f" Lancement de l'entraînement Classifier (Cross-Entropy Loss) pour {EPOCHS} époques...")

for epoch in range(EPOCHS + 1):

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(images)
        preds = torch.argmax(logits, dim=1)
        is_correct = (preds == labels).cpu().numpy()

    for i in range(len(embeddings)):
        embedding_cpu = embeddings[i].cpu().numpy()
        p_idx = preds[i].item()
        l_idx = labels[i].item()
        correct_status = "Correct" if is_correct[i] else "Erreur"

        history_df.append({
            'Epoch': f"Époque {epoch}",
            'X': embedding_cpu[0],
            'Y': embedding_cpu[1],
            'Vraie Classe': class_names[l_idx],
            'Classe Prédite': class_names[p_idx],
            'Correct': correct_status
        })

    if epoch == EPOCHS: break

    model.train()
    optimizer.zero_grad()

    logits, _ = model(images)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        with torch.no_grad():
            train_acc = (torch.argmax(logits, 1) == labels).float().mean()
        print(f"Epoch {epoch}/{EPOCHS} - Loss: {loss.item():.4f} - Accuracy: {train_acc:.1%}")

print(" Entraînement Classifier terminé !")

df_plot = pd.DataFrame(history_df)

padding = 1.0
ranges = {
    'x': [df_plot['X'].min() - padding, df_plot['X'].max() + padding],
    'y': [df_plot['Y'].min() - padding, df_plot['Y'].max() + padding]
}

fig = px.scatter(
    df_plot,
    x='X', y='Y',
    color='Vraie Classe',
    symbol='Correct',
    symbol_sequence=['circle', 'x'],
    animation_frame='Epoch',
    title="Évolution de l'Espace Latent (CNN Classifier - Cross Entropy Loss)",
    range_x=ranges['x'], range_y=ranges['y'],
    hover_data={'Classe Prédite': True, 'Correct': True, 'X': ':.2f', 'Y': ':.2f'},
    opacity=0.8
)

fig.update_traces(marker=dict(size=6, line=dict(width=1, color='DarkSlateGrey')))
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 500

print(" Génération de l'animation... Cela peut prendre quelques secondes.")
fig.show()

## Vrai test sur dataset Cars


In [ ]:
from pathlib import Path

data_dir = Path("../data/raw/Cars")

queries_to_exclude = {
    "0_1_BMW_X3_207.jpg", "0_0_BMW_Serie3Berline_74.jpg", "0_2_BMW_i8_299.jpg",
    "2_0_Volkswagen_Touareg_2822.jpg", "2_4_Volkswagen_Polo_3463.jpg", "2_9_Volkswagen_T-Roc_4209.jpg",
    "4_2_Opel_vivarofourgon_5999.jpg", "4_4_Opel_Insignatourer_6353.jpg", "4_9_Opel_zafiralife_6887.jpg",
    "6_0_Hyundai_Nexo_8282.jpg", "6_3_Hyundai_i10_8837.jpg", "6_5_Hyundai_i30_9125.jpg",
    "8_1_Ford_Puma_11276.jpg", "8_5_Ford_Explorer_11897.jpg", "8_6_Ford_Focus_11951.jpg"
}

all_image_paths = [f for f in data_dir.glob("*.jpg") if f.name not in queries_to_exclude]
all_labels = []
number_labels = set()

print(f"Nombre total d'images après exclusion : {len(all_image_paths)}")

for img_path in all_image_paths:
    parts = img_path.stem.split('_')
    img_class = f"{parts[0]}_{parts[1]}"
    all_labels.append(img_class)
    if img_class not in number_labels:
        number_labels.add(img_class)

print(f"Nombre total de labels après exclusion : {len(all_labels)}")
print(f"Nombre de labels uniques : {len(number_labels)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedKFold
import numpy as np
import random
from PIL import Image

class TripletCarsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
        self.model_to_indices = {label: [] for label in set(labels)}
        for idx, label in enumerate(labels):
            self.model_to_indices[label].append(idx)
            
        self.brand_to_models = {}
        for label in set(labels):
            brand = label.split('_')[0]
            if brand not in self.brand_to_models:
                self.brand_to_models[brand] = []
            self.brand_to_models[brand].append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        anchor_path = self.image_paths[index]
        anchor_label = self.labels[index]
        anchor_brand = anchor_label.split('_')[0]
        
        positive_index = index
        while positive_index == index:
            positive_index = random.choice(self.model_to_indices[anchor_label])
        positive_path = self.image_paths[positive_index]
        
        is_hard_negative = random.random() < 0.5
        
        if is_hard_negative and len(self.brand_to_models[anchor_brand]) > 1:
            negative_label = anchor_label
            while negative_label == anchor_label:
                negative_label = random.choice(self.brand_to_models[anchor_brand])
        else:
            negative_brand = anchor_brand
            while negative_brand == anchor_brand:
                negative_brand = random.choice(list(self.brand_to_models.keys()))
            negative_label = random.choice(self.brand_to_models[negative_brand])
            
        negative_index = random.choice(self.model_to_indices[negative_label])
        negative_path = self.image_paths[negative_index]
        
        img_a = Image.open(anchor_path).convert('RGB')
        img_p = Image.open(positive_path).convert('RGB')
        img_n = Image.open(negative_path).convert('RGB')
        
        if self.transform:
            img_a = self.transform(img_a)
            img_p = self.transform(img_p)
            img_n = self.transform(img_n)
            
        return img_a, img_p, img_n

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Grayscale(num_output_channels=3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

X = np.array(all_image_paths)
y = np.array(all_labels)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
import torch.nn as nn
from torchvision import models

class EfficientNetMetricLearning(nn.Module):
    def __init__(self, embedding_size=512):
        super(EfficientNetMetricLearning, self).__init__()
        
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        
        in_features = self.backbone.classifier[1].in_features 
        
        self.backbone.classifier = nn.Sequential(
            nn.Linear(in_features, embedding_size)
        )
        
    def forward(self, x):
        x = self.backbone(x)
        
        x = nn.functional.normalize(x, p=2, dim=1)
        
        return x


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from dotenv import load_dotenv
import wandb

load_dotenv()
wandb.login(key=os.getenv("WANDB_API_KEY"))

def train_metric_learning_model(model, train_loader, val_loader, fold_idx=1, num_epochs=50, patience=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.TripletMarginLoss(margin=1.0, p=2)
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4) 
    
    wandb.init(
        project="MIR_Cars_Project",
        name=f"EfficientNet-B0_Fold_{fold_idx}",
        config={
            "architecture": "EfficientNet-B0",
            "epochs": num_epochs,
            "batch_size": train_loader.batch_size,
            "learning_rate": 0.0001,
            "margin": 1.0,
            "patience": patience
        }
    )
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_path = f"best_model_fold_{fold_idx}.pth"
    
    print(f"\nDébut de l'entraînement sur {device} (Fold {fold_idx})...")
    
    for epoch in range(num_epochs):
        model.train()
        running_train_loss = 0.0
        
        for batch_idx, (anchor, positive, negative) in enumerate(train_loader):
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            
            optimizer.zero_grad()
            
            emb_a = model(anchor)
            emb_p = model(positive)
            emb_n = model(negative)
            
            loss = criterion(emb_a, emb_p, emb_n)
            
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        model.eval()
        running_val_loss = 0.0
        
        with torch.no_grad():
            for anchor, positive, negative in val_loader:
                anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
                
                emb_a = model(anchor)
                emb_p = model(positive)
                emb_n = model(negative)
                
                loss = criterion(emb_a, emb_p, emb_n)
                running_val_loss += loss.item()
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss
        })
        
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"   Validation loss améliorée. Modèle sauvegardé sous '{best_model_path}'")
        else:
            epochs_no_improve += 1
            print(f"   Pas d'amélioration depuis {epochs_no_improve} epoch(s).")
            
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping déclenché à l'epoch {epoch+1} !")
                break

    wandb.finish()
    print(f"Entraînement terminé ! Le meilleur modèle a une Val Loss de {best_val_loss:.4f}")
    
    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    return model, best_val_loss

In [ ]:
import shutil

global_best_val_loss = float('inf')
global_best_model_path = "best_model_GLOBAL.pth"
best_fold = -1

print("DÉBUT DE LA VALIDATION CROISÉE K-FOLD (5 Folds)")

for fold_idx, (train_index, val_index) in enumerate(skf.split(X, y), 1):
    print(f"\n=========================================")
    print(f"          LANCEMENT DU FOLD {fold_idx}/5")
    print(f"=========================================")
    
    X_train, y_train = X[train_index], y[train_index]
    X_val, y_val = X[val_index], y[val_index]
    
    train_dataset = TripletCarsDataset(X_train, y_train, transform=train_transforms)
    val_dataset = TripletCarsDataset(X_val, y_val, transform=val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
    
    model_for_this_fold = EfficientNetMetricLearning(embedding_size=512)
    
    trained_model, fold_best_val_loss = train_metric_learning_model(
        model=model_for_this_fold,
        train_loader=train_loader,
        val_loader=val_loader,
        fold_idx=fold_idx,
        num_epochs=100,
        patience=10 
    )
    
    if fold_best_val_loss < global_best_val_loss:
        print(f"\nNOUVEAU CHAMPION GLOBAL ! Le Fold {fold_idx} bat le record avec une Val Loss de {fold_best_val_loss:.4f}")
        global_best_val_loss = fold_best_val_loss
        best_fold = fold_idx
        
        fold_model_path = f"best_model_fold_{fold_idx}.pth"
        shutil.copy(fold_model_path, global_best_model_path)

print(f"\n=========================================")
print(f"FIN DU K-FOLD !")
print(f"Le meilleur modèle absolu provient du Fold {best_fold} (Val Loss: {global_best_val_loss:.4f})")
print(f"Il est sauvegardé sous : {global_best_model_path}")
print(f"=========================================")

In [ ]:
import time 

class SimpleCarsDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img = Image.open(self.image_paths[index]).convert('RGB')
        label = self.labels[index]
        if self.transform:
            img = self.transform(img)
        return img, label

query_paths = [os.path.join(data_dir, q) for q in queries_to_exclude]
query_labels = []
for q in queries_to_exclude:
    parts = q.split('_')
    query_labels.append(f"{parts[0]}_{parts[1]}")

gallery_dataset = SimpleCarsDataset(X, y, transform=val_transforms)
query_dataset = SimpleCarsDataset(query_paths, query_labels, transform=val_transforms)

gallery_loader = DataLoader(gallery_dataset, batch_size=32, shuffle=False)
query_loader = DataLoader(query_dataset, batch_size=15, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EfficientNetMetricLearning(embedding_size=512)
model.load_state_dict(torch.load("best_model_GLOBAL.pth"))
model = model.to(device)
model.eval()

def extract_features(loader):
    features = []
    labels = []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            out = model(imgs)
            features.append(out.cpu().numpy())
            labels.extend(lbls)
    return np.vstack(features), np.array(labels)

print(" Début de l'indexation de la base de données...")
start_time = time.time()
gallery_features, gallery_labels_arr = extract_features(gallery_loader)
indexing_time = time.time() - start_time

descriptor_size_mb = (gallery_features.size * gallery_features.itemsize) / (1024 * 1024)

print(f" Temps d'indexation (Indexing Time) : {indexing_time:.2f} s")
print(f" Taille des descripteurs (Descr. size) : {descriptor_size_mb:.2f} MB")

query_features, query_labels_arr = extract_features(query_loader)

print(" Début de la recherche...")
start_time = time.time()
similarities = np.dot(query_features, gallery_features.T)
search_time = time.time() - start_time
avg_search_time = search_time / len(query_features)

print(f" Temps moyen de recherche par image : {avg_search_time:.5f} s")

def calculate_metrics(similarities, query_labels, gallery_labels, top_k):
    sorted_indices = np.argsort(similarities, axis=1)[:, ::-1]
    
    recalls, precisions, aps = [], [], []
    
    for i in range(len(query_labels)):
        q_label = query_labels[i]
        
        top_k_indices = sorted_indices[i, :top_k]
        top_k_labels = gallery_labels[top_k_indices]
        
        hits = (top_k_labels == q_label).astype(int)
        
        total_relevant_in_gallery = np.sum(gallery_labels == q_label)
        
        relevant_retrieved = np.sum(hits)
        recall = relevant_retrieved / total_relevant_in_gallery
        precision = relevant_retrieved / top_k
        
        ap = 0.0
        relevant_so_far = 0
        for rank, is_hit in enumerate(hits):
            if is_hit:
                relevant_so_far += 1
                ap += relevant_so_far / (rank + 1.0)
        if relevant_retrieved > 0:
            ap /= min(total_relevant_in_gallery, top_k)
            
        recalls.append(recall)
        precisions.append(precision)
        aps.append(ap)
        
    return np.array(recalls), np.array(precisions), np.array(aps)

r_50, p_50, ap_50 = calculate_metrics(similarities, query_labels_arr, gallery_labels_arr, top_k=50)
r_100, p_100, ap_100 = calculate_metrics(similarities, query_labels_arr, gallery_labels_arr, top_k=100)

map_50 = np.mean(ap_50)
map_100 = np.mean(ap_100)

print("\n RÉSULTATS POUR LE TABLEAU 4 :")
for i, q_label in enumerate(query_labels_arr):
    print(f"Requête R{i+1} (Classe {q_label}) :")
    print(f"   Top 50  -> R: {r_50[i]:.4f} | P: {p_50[i]:.4f} | AP: {ap_50[i]:.4f}")
    print(f"   Top 100 -> R: {r_100[i]:.4f} | P: {p_100[i]:.4f} | AP: {ap_100[i]:.4f}")

print(f"\n mAP@50  : {map_50:.4f}")
print(f" mAP@100 : {map_100:.4f}")

In [ ]:
import matplotlib.pyplot as plt

def show_top_results(query_idx, top_k=5):
    q_img_path = query_dataset.image_paths[query_idx]
    q_label = query_labels_arr[query_idx]
    
    sorted_indices = np.argsort(similarities[query_idx])[::-1]
    top_indices = sorted_indices[:top_k]
    
    fig, axes = plt.subplots(1, top_k + 1, figsize=(15, 4))
    
    axes[0].imshow(Image.open(q_img_path))
    axes[0].set_title(f"REQUÊTE\nClasse {q_label}", color='blue')
    axes[0].axis('off')
    
    for i, idx in enumerate(top_indices):
        res_img_path = gallery_dataset.image_paths[idx]
        res_label = gallery_labels_arr[idx]
        
        axes[i+1].imshow(Image.open(res_img_path))
        color = 'green' if res_label == q_label else 'red'
        axes[i+1].set_title(f"Top {i+1}\nClasse {res_label}", color=color)
        axes[i+1].axis('off')
        
    plt.tight_layout()
    plt.show()

show_top_results(0)
show_top_results(1)
show_top_results(2)